# مشروع استخراج وتصحيح نصوص الخط اليدوي

### HandwrittenOCR v3.0
واجهة ويب تفاعلية + دعم Colab Notebook

## الخطوات:
1. تثبيت المكتبات
2. إعداد المسارات والنماذج
3. تحميل النماذج (TrOCR + EasyOCR)
4. معالجة PDF واستخراج النصوص
5. مراجعة الكلمات والجمل
6. تصدير البيانات وتحسين النموذج
7. تشغيل خادم API للواجهة الويب

In [ ]:
# تثبيت جميع المكتبات المطلوبة
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image easyocr pyspellchecker langdetect pytesseract transformers torch torchvision Pillow opencv-python-headless pandas ar-corrector ipywidgets scikit-learn
!pip install -q peft huggingface_hub datasets fastapi uvicorn python-multipart pyngrok

In [ ]:
from google.colab import userdata
import os

# جلب التوكن المخزن في Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("تم تحميل Hugging Face Token من Colab Secrets")
except userdata.SecretNotFoundError:
    print("لم يتم العثور على Secret باسم 'HF_TOKEN'.")
except Exception as e:
    print(f"خطأ: {e}")

import cv2, numpy as np, sqlite3, io, torch, json, time, logging
import pandas as pd, easyocr
from PIL import Image
from pdf2image import convert_from_path
from google.colab import drive, patches
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from ar_corrector.corrector import Corrector
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

# ربط Google Drive
drive.mount('/content/drive')

# إعداد المسارات
MODEL_CACHE = '/content/drive/MyDrive/Handwriting_Dataset/models_cache'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE
os.environ["TORCH_HOME"] = MODEL_CACHE

PDF_PATH = '/content/drive/MyDrive/python notes.pdf'
OUTPUT_DIR = '/content/drive/MyDrive/Handwriting_Dataset'
LOGS_DIR = os.path.join(OUTPUT_DIR, 'Logs')
os.makedirs(LOGS_DIR, exist_ok=True)

DB_PATH = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
LOG_FILE = os.path.join(LOGS_DIR, f"ocr_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
FEEDBACK_CSV = os.path.join(LOGS_DIR, "user_corrections_feedback.csv")
STATS_JSON = os.path.join(LOGS_DIR, "processing_stats.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'ocr_checkpoint.json')
CORRECTION_DICT_PATH = os.path.join(OUTPUT_DIR, 'correction_dict.json')

# إعداد logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE, encoding='utf-8'), logging.StreamHandler()]
)

if not os.path.exists(FEEDBACK_CSV):
    pd.DataFrame(columns=['timestamp', 'image_id', 'original_text', 'corrected_text', 'status']).to_csv(FEEDBACK_CSV, index=False, encoding='utf-8')

print("بدء تشغيل المشروع")


In [ ]:
print("جاري فحص النماذج المحملة مسبقاً...")
start_time = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"الجهاز: {device}")

# TrOCR
try:
    trocr_processor = TrOCRProcessor.from_pretrained("David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE)
    trocr_model = VisionEncoderDecoderModel.from_pretrained("David-Magdy/TR_OCR_LARGE", cache_dir=MODEL_CACHE).to(device)
    print("TrOCR جاهز")
except Exception as e:
    print(f"فشل تحميل TrOCR: {e}")

# تحميل LoRA إذا كان موجوداً
lora_save_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
if os.path.exists(lora_save_path):
    from peft import PeftModel
    try:
        trocr_model = PeftModel.from_pretrained(trocr_model, lora_save_path).to(device)
        print("تم تحميل النموذج المحسن (LoRA)")
    except Exception as e:
        print(f"فشل تحميل LoRA: {e}. سيتم استخدام النموذج الأساسي.")
else:
    print("يستخدم النموذج الأساسي (لا يوجد fine-tuning بعد)")

# EasyOCR
easy_reader = easyocr.Reader(['en', 'ar'])

# المدققات الإملائية
arabic_spell = Corrector()
english_spell = SpellChecker(language='en')
DetectorFactory.seed = 0

print(f"تم التحميل في {time.time()-start_time:.2f} ثانية")

# EasyOCR على Drive
import os.path as osp, shutil
drive_easyocr_path = '/content/drive/MyDrive/Handwriting_Dataset/.EasyOCR'
local_easyocr_path = os.path.expanduser('~/.EasyOCR')

if not os.path.exists(drive_easyocr_path) and os.path.exists(local_easyocr_path):
    print('جاري نقل نماذج EasyOCR إلى Drive...')
    shutil.move(local_easyocr_path, drive_easyocr_path)

if not os.path.islink(local_easyocr_path):
    if os.path.exists(local_easyocr_path):
        shutil.rmtree(local_easyocr_path)
    os.symlink(drive_easyocr_path, local_easyocr_path)
    print('تم ربط نماذج EasyOCR بـ Drive')


In [ ]:
import json as _json

def save_checkpoint(page_num, total_pages, processed_words):
    checkpoint = {'last_page_processed': page_num, 'total_pages': total_pages, 'processed_words': processed_words, 'timestamp': datetime.now().isoformat()}
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        _json.dump(checkpoint, f, indent=2, ensure_ascii=False)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return _json.load(f)
    return None

def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

def build_correction_dict():
    if not os.path.exists(FEEDBACK_CSV): return {}
    try:
        df_fb = pd.read_csv(FEEDBACK_CSV, encoding='utf-8')
        corrections = {}
        for _, row in df_fb.iterrows():
            orig = str(row['original_text']).strip()
            corr = str(row['corrected_text']).strip()
            if orig and corr and orig != corr:
                if orig not in corrections: corrections[orig] = {}
                corrections[orig][corr] = corrections[orig].get(corr, 0) + 1
        final_dict = {}
        for orig, candidates in corrections.items():
            best = max(candidates, key=candidates.get)
            if candidates[best] >= 2:
                final_dict[orig] = best
        with open(CORRECTION_DICT_PATH, 'w', encoding='utf-8') as f:
            _json.dump(final_dict, f, ensure_ascii=False, indent=2)
        return final_dict
    except:
        return {}

def apply_correction_dict(text, correction_dict):
    if not correction_dict or not text: return text
    words = text.split()
    return ' '.join([correction_dict.get(w, w) for w in words])

def preprocess_image(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary

def smart_word_segmentation(img_bgr, binary_img, existing_detections=None):
    word_boxes = []
    if existing_detections:
        for (bbox, text, conf) in existing_detections:
            pts = np.array(bbox, dtype=np.int32)
            x, y, w, h = cv2.boundingRect(pts)
            word_boxes.append((x, y, w, h))
        return word_boxes
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25,5))
    dilated = cv2.dilate(binary_img, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if bw > 15 and bh > 10: word_boxes.append((x, y, bw, bh))
    return word_boxes

def recognize_word_ensemble(img_bgr, easyocr_raw=None):
    results = []
    try:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        inputs = trocr_processor(images=Image.fromarray(rgb), return_tensors="pt").pixel_values.to(device)
        generated_ids = trocr_model.generate(inputs, max_length=50)
        trocr_text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        results.append(('trocr', trocr_text, 0.7))
    except: pass
    if easyocr_raw:
        results.append(('easyocr', easyocr_raw[1], easyocr_raw[2]))
    if not results: return "", 0.0, "none", True
    best = max(results, key=lambda x: x[2])
    return best[1], best[2], best[0], (best[2] < 0.5)

def correct_text(text):
    if not text: return ""
    try:
        lang = detect(text)
        if lang == 'ar': return arabic_spell.contextual_correct(text)
        else:
            words = text.split()
            corrected = [english_spell.correction(w) or w for w in words]
            return ' '.join(corrected)
    except: return text


In [ ]:
def process_pdf(pages_start=1, pages_end=2, resume=True):
    proc_start = time.time()
    checkpoint = load_checkpoint() if resume else None
    if checkpoint:
        print(f"استئناف من الصفحة {checkpoint['last_page_processed']}")
        pages_start = checkpoint['last_page_processed']

    correction_dict = build_correction_dict()
    try:
        images = convert_from_path(PDF_PATH, dpi=300, first_page=pages_start, last_page=pages_end)
    except Exception as e:
        print(f"فشل معالجة PDF: {e}"); return

    total_words_processed = checkpoint.get('processed_words', 0) if checkpoint else 0
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
                        (image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                         image_data BLOB, predicted_text TEXT, status TEXT,
                         confidence REAL, model_source TEXT,
                         x INTEGER, y INTEGER, w INTEGER, h INTEGER, page_num INTEGER)''')

        for idx, pil in enumerate(images):
            actual_page = pages_start + idx
            img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
            try: detections = easy_reader.readtext(img_bgr, detail=1)
            except: detections = []

            binary = preprocess_image(img_bgr)
            boxes_info = []
            if detections:
                for (bbox, text, conf) in detections:
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    boxes_info.append(((x,y,w,h), (bbox, text, conf)))
            else:
                manual_boxes = smart_word_segmentation(img_bgr, binary)
                for box in manual_boxes: boxes_info.append((box, None))

            for (box, e_raw) in boxes_info:
                x, y, w, h = box
                crop = img_bgr[y:y+h, x:x+w]
                raw, conf, src, _ = recognize_word_ensemble(crop, easyocr_raw=e_raw)
                final = apply_correction_dict(correct_text(raw), correction_dict)
                _, buf = cv2.imencode(".png", crop)
                conn.execute('''INSERT INTO handwriting_data
                                (image_data, predicted_text, status, confidence, model_source, x, y, w, h, page_num)
                                VALUES (?,?,?,?,?,?,?,?,?,?)''',
                             (buf.tobytes(), final, 'unverified', conf, src, x, y, w, h, actual_page))
                total_words_processed += 1

            save_checkpoint(actual_page + 1, pages_end, total_words_processed)
            patches.cv2_imshow(cv2.resize(img_bgr, (0,0), fx=0.3, fy=0.3))
        conn.commit()

    clear_checkpoint()
    stats = {"timestamp": datetime.now().isoformat(), "pages": pages_end-pages_start+1, "words": total_words_processed}
    with open(STATS_JSON, 'w', encoding='utf-8') as f: _json.dump(stats, f, ensure_ascii=False)
    print(f"اكتملت المعالجة في {time.time()-proc_start:.2f} ثانية.")


## تشغيل عملية الاستخراج

In [ ]:
process_pdf(pages_start=1, pages_end=2, resume=True)

In [ ]:
from IPython.display import clear_output

def launch_review_ui_v2():
    """واجهة مراجعة الكلمات - Jupyter v3"""
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY confidence ASC",
            conn
        )
    if df.empty:
        print("لا توجد كلمات جديدة للمراجعة.")
        return

    current = [0]
    img = widgets.Image(format='png', width=350)
    text = widgets.Text(description="النص:")
    conf_bar = widgets.FloatProgress(min=0, max=1.0, description='الثقة:', layout={'width': '95%'})
    conf_label = widgets.HTML(value="")
    prog = widgets.IntProgress(min=0, max=len(df)-1, bar_style='info')
    info = widgets.Label()

    def update():
        if 0 <= current[0] < len(df):
            row = df.iloc[current[0]]
            img.value = row['image_data']
            text.value = str(row['predicted_text'] or '')
            c = float(row['confidence'])
            conf_bar.value = c
            conf_bar.bar_style = 'danger' if c < 0.5 else ('warning' if c < 0.8 else 'success')
            conf_label.value = f"<b>{c:.2%}</b>"
            info.value = f"{current[0]+1}/{len(df)} | صفحة {row['page_num']}"
            prog.value = current[0]
        else:
            img.value = b''; text.value = ''; info.value = "اكتملت المراجعة"

    def on_confirm(b):
        if not (0 <= current[0] < len(df)): return
        rid = int(df.iloc[current[0]]['image_id'])
        original = df.iloc[current[0]]['predicted_text']
        corrected = text.value
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("UPDATE handwriting_data SET predicted_text=?, status='verified' WHERE image_id=?", (corrected, rid))
        if original != corrected:
            pd.DataFrame([{
                'timestamp': datetime.now().isoformat(),
                'image_id': rid,
                'original_text': original,
                'corrected_text': corrected,
                'status': 'verified'
            }]).to_csv(FEEDBACK_CSV, mode='a', header=not os.path.exists(FEEDBACK_CSV), index=False, encoding='utf-8')
        df.drop(df.index[current[0]], inplace=True)
        df.reset_index(drop=True, inplace=True)
        prog.max = max(0, len(df)-1)
        if current[0] >= len(df) and len(df) > 0: current[0] = len(df) - 1
        update()

    def on_next(b):
        current[0] = min(len(df)-1, current[0] + 1)
        update()

    def on_prev(b):
        current[0] = max(0, current[0] - 1)
        update()

    def on_delete(b):
        if not (0 <= current[0] < len(df)): return
        rid = int(df.iloc[current[0]]['image_id'])
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute('DELETE FROM handwriting_data WHERE image_id=?', (rid,))
        df.drop(df.index[current[0]], inplace=True)
        df.reset_index(drop=True, inplace=True)
        prog.max = max(0, len(df)-1)
        if current[0] >= len(df) and len(df) > 0: current[0] = len(df)-1
        update()

    btn_confirm = widgets.Button(description="تأكيد", button_style='success')
    btn_next = widgets.Button(description="التالي", button_style='info')
    btn_prev = widgets.Button(description="السابق", button_style='info')
    btn_del = widgets.Button(description="حذف", button_style='danger')
    btn_confirm.on_click(on_confirm)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_del.on_click(on_delete)

    display(widgets.VBox([
        prog, info, img, text,
        widgets.HBox([conf_bar, conf_label]),
        widgets.HBox([btn_prev, btn_confirm, btn_del, btn_next])
    ]))
    update()

launch_review_ui_v2()


In [ ]:
from PIL import Image as PILImage
import io as _io

def launch_sentence_review_ui(y_tolerance=25):
    """واجهة مراجعة الجمل في Jupyter/Colab"""
    with sqlite3.connect(DB_PATH) as conn:
        words_df = pd.read_sql_query(
            "SELECT * FROM handwriting_data ORDER BY page_num, y, x", conn
        )

    if words_df.empty:
        print("لا توجد بيانات للمراجعة.")
        return

    # تجميع الجمل
    sentences = []
    for page in words_df['page_num'].unique():
        p_words = words_df[words_df['page_num'] == page].copy()
        if p_words.empty:
            continue
        curr_line = [p_words.iloc[0].to_dict()]
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                curr_line.append(row)
            else:
                sentences.append(curr_line)
                curr_line = [row]
        sentences.append(curr_line)

    current_idx = [0]

    img_container = widgets.HBox(layout={'overflow_x': 'scroll', 'padding': '10px'})
    sentence_input = widgets.Textarea(description="الجملة:", layout={'width': '95%', 'height': '80px'})
    info_label = widgets.Label()
    progress = widgets.IntProgress(min=0, max=len(sentences)-1, layout={'width': '95%', 'height': '20px'})

    def get_img_widget(blob):
        img = PILImage.open(_io.BytesIO(blob))
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        return widgets.Image(value=buf.getvalue(), format='png', width=120)

    def update_ui():
        if not (0 <= current_idx[0] < len(sentences)):
            return
        sent = sentences[current_idx[0]]
        img_container.children = [get_img_widget(w['image_data']) for w in sent]
        original_text = " ".join([str(w['predicted_text'] or "") for w in sent])
        sentence_input.value = original_text
        info_label.value = f"جملة {current_idx[0]+1} من {len(sentences)} | صفحة {sent[0]['page_num']}"
        progress.value = current_idx[0]

    def save_current(b):
        sent = sentences[current_idx[0]]
        original = " ".join([str(w['predicted_text'] or "") for w in sent])
        corrected = sentence_input.value.strip()
        if not corrected:
            return

        # تحديث حالة الكلمات
        with sqlite3.connect(DB_PATH) as conn:
            for w in sent:
                conn.execute(
                    "UPDATE handwriting_data SET status='sentence_corrected' WHERE image_id=?",
                    (w['image_id'],)
                )

        # تسجيل تصحيح الجملة
        sent_id = f"p{sent[0]['page_num']}_y{sent[0]['y']}"
        if original != corrected:
            pd.DataFrame([{
                'timestamp': datetime.now().isoformat(),
                'image_id': None,
                'original_text': original,
                'corrected_text': corrected,
                'status': f'sent_rev_{sent_id}'
            }]).to_csv(FEEDBACK_CSV, mode='a', header=not os.path.exists(FEEDBACK_CSV), index=False, encoding='utf-8')

            # استخراج تصحيحات مشتقة على مستوى الكلمات
            orig_words = original.split()
            corr_words = corrected.split()
            if len(orig_words) == len(corr_words):
                derived = []
                for o, c in zip(orig_words, corr_words):
                    if o != c:
                        derived.append({
                            'timestamp': datetime.now().isoformat(),
                            'image_id': None,
                            'original_text': o,
                            'corrected_text': c,
                            'status': 'sentence_derived'
                        })
                if derived:
                    pd.DataFrame(derived).to_csv(
                        FEEDBACK_CSV, mode='a',
                        header=False, index=False, encoding='utf-8'
                    )

        print(f"تم حفظ الجملة {current_idx[0]+1}")
        current_idx[0] = min(len(sentences)-1, current_idx[0] + 1)
        update_ui()

    def on_next(b):
        current_idx[0] = min(len(sentences)-1, current_idx[0] + 1)
        update_ui()

    def on_prev(b):
        current_idx[0] = max(0, current_idx[0] - 1)
        update_ui()

    btn_save = widgets.Button(description="حفظ وتأكيد", button_style='success')
    btn_next = widgets.Button(description="التالي", button_style='info')
    btn_prev = widgets.Button(description="السابق", button_style='info')
    btn_save.on_click(save_current)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)

    display(widgets.VBox([
        progress, info_label, img_container,
        sentence_input,
        widgets.HBox([btn_prev, btn_save, btn_next])
    ]))
    update_ui()

launch_sentence_review_ui()


In [ ]:
import random

# ---- تصدير بيانات التدريب ----
def export_finetuning_dataset(output_dir=None, val_ratio=0.1):
    if output_dir is None:
        output_dir = os.path.join(OUTPUT_DIR, 'hf_training_dataset')
    os.makedirs(output_dir, exist_ok=True)
    img_dir = os.path.join(output_dir, "images")
    os.makedirs(img_dir, exist_ok=True)

    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status IN ('verified', 'sentence_corrected')",
            conn
        )
    if df_v.empty:
        print("لا توجد بيانات موثقة."); return None

    records = []
    for _, row in df_v.iterrows():
        fname = f"img_{row['image_id']}.png"
        with open(os.path.join(img_dir, fname), "wb") as f:
            f.write(row['image_data'])
        text = (str(row['predicted_text']) or "").strip()
        if text:
            records.append({"image": fname, "text": text, "verified": True})

    if not records:
        print("لا توجد سجلات صالحة."); return None

    random.shuffle(records)
    split = int(len(records) * (1 - val_ratio))
    for data, name in [(records[:split], "train.jsonl"), (records[split:], "val.jsonl")]:
        with open(os.path.join(output_dir, name), "w", encoding="utf-8") as f:
            for r in data:
                f.write(_json.dumps(r, ensure_ascii=False) + "\n")
    print(f"تم التصدير: {len(records)} عينة (train={len(records[:split])}, val={len(records[split:])})")
    print(f"المسار: {output_dir}")
    return output_dir


# ---- رفع إلى HuggingFace ----
def push_to_huggingface(local_dataset_dir, hf_repo_id):
    from huggingface_hub import HfApi, login
    if HF_TOKEN:
        login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(repo_id=hf_repo_id, repo_type="dataset", exist_ok=True)
    api.upload_folder(
        folder_path=local_dataset_dir,
        repo_id=hf_repo_id,
        repo_type="dataset",
        commit_message=f"Update dataset - {datetime.now().strftime('%Y-%m-%d')}"
    )
    print(f"تم الرفع بنجاح: https://huggingface.co/datasets/{hf_repo_id}")


# ---- تدريب LoRA على TrOCR ----
def finetune_trocr_lora(min_samples=100, epochs=3, batch_size=4, lr=5e-5):
    from peft import get_peft_model, LoraConfig, TaskType
    from torch.optim import AdamW
    from torch.utils.data import Dataset, DataLoader

    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query(
            "SELECT image_data, predicted_text FROM handwriting_data WHERE status IN ('verified', 'sentence_corrected')",
            conn
        )
    if len(df_v) < min_samples:
        print(f"لديك {len(df_v)} عينة فقط. الحد الأدنى: {min_samples}.")
        return

    print(f"بدء التدريب على {len(df_v)} عينة...")

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16, lora_alpha=32,
        target_modules=["query", "value"],
        lora_dropout=0.1
    )
    model = get_peft_model(trocr_model, lora_config)
    model.train()

    class HandwritingDataset(Dataset):
        def __init__(self, df):
            self.df = df
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            pv = trocr_processor(images=img, return_tensors="pt").pixel_values.squeeze()
            labels = trocr_processor.tokenizer(
                row['predicted_text'], return_tensors="pt",
                padding="max_length", max_length=50
            ).input_ids.squeeze()
            labels[labels == trocr_processor.tokenizer.pad_token_id] = -100
            return {"pixel_values": pv, "labels": labels}

    loader = DataLoader(HandwritingDataset(df_v), batch_size=batch_size, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        batch_count = 0
        for batch in loader:
            out = model(
                pixel_values=batch["pixel_values"].to(device),
                labels=batch["labels"].to(device)
            )
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += out.loss.item()
            batch_count += 1
        avg_loss = total_loss / max(batch_count, 1)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    save_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
    model.save_pretrained(save_path)
    trocr_processor.save_pretrained(save_path)
    print(f"تم حفظ النموذج المحسن في: {save_path}")


# ---- إعادة تجميع الجمل ----
def reconstruct_sentences(y_tolerance=25):
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status IN ('verified', 'sentence_corrected') ORDER BY page_num, y, x",
            conn
        )
    if words.empty:
        return None
    all_s = []
    for page in words['page_num'].unique():
        pw = words[words['page_num'] == page].copy()
        lines, curr = [], [pw.iloc[0].to_dict()]
        for i in range(1, len(pw)):
            r = pw.iloc[i].to_dict()
            if abs(r['y'] - curr[-1]['y']) <= y_tolerance:
                curr.append(r)
            else:
                lines.append(curr)
                curr = [r]
        lines.append(curr)
        for line in lines:
            preview = ' '.join([str(w['predicted_text']) for w in line])
            try:
                lang = detect(preview)
            except:
                lang = 'en'
            sl = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            all_s.append({
                'page': page,
                'text': ' '.join([str(w['predicted_text']) for w in sl]),
                'lang': lang
            })
    return pd.DataFrame(all_s)


print("✅ تم تحميل دوال التصدير والتدريب والرفع")
print(f"   تصدير:  export_finetuning_dataset()")
print(f"   رفع:    push_to_huggingface(local_dataset_dir, hf_repo_id)")
print(f"   تدريب:  finetune_trocr_lora(min_samples=100)")
print(f"   تجميع:  reconstruct_sentences()")


In [ ]:
def edit_correction_dict_ui():
    """واجهة تعديل قاموس التصحيحات يدوياً"""
    data = {}
    if os.path.exists(CORRECTION_DICT_PATH):
        with open(CORRECTION_DICT_PATH, 'r', encoding='utf-8') as f:
            data = json.load(f)

    out = widgets.Output()
    key_in = widgets.Text(description="أصلي:")
    val_in = widgets.Text(description="تصحيح:")
    status_label = widgets.Label(value=f"عدد التصحيحات: {len(data)}")

    # عرض التصحيحات الحالية
    list_output = widgets.Output()
    with list_output:
        if data:
            print("التصحيحات الحالية:")
            for k, v in list(data.items())[:50]:
                print(f"  '{k}' → '{v}'")
            if len(data) > 50:
                print(f"  ... و {len(data)-50} تصحيح آخر")
        else:
            print("القاموس فارغ.")

    def save(b):
        if key_in.value:
            data[key_in.value] = val_in.value
            with open(CORRECTION_DICT_PATH, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
            status_label.value = f"عدد التصحيحات: {len(data)}"
            with out:
                clear_output()
                print(f"تم حفظ: '{key_in.value}' → '{val_in.value}'")

    def rebuild(b):
        nonlocal data
        data = build_correction_dict()
        status_label.value = f"عدد التصحيحات: {len(data)}"
        with out:
            clear_output()
            print(f"تم إعادة بناء القاموس: {len(data)} تصحيح")

    btn_save = widgets.Button(description="حفظ", button_style='success')
    btn_rebuild = widgets.Button(description="إعادة بناء من التصحيحات", button_style='info')
    btn_save.on_click(save)
    btn_rebuild.on_click(rebuild)

    display(widgets.VBox([status_label, key_in, val_in, widgets.HBox([btn_save, btn_rebuild]), out, list_output]))

edit_correction_dict_ui()


In [ ]:
# ==========================================
# تشغيل خادم API للواجهة الويب
# ==========================================

# Download the FastAPI backend code from GitHub
!rm -rf /content/HandwrittenOCR_web
!git clone https://github.com/DrAbdulmalek/HandwrittenOCR.git /content/HandwrittenOCR_web --depth 1

import sys
sys.path.insert(0, '/content/HandwrittenOCR_web')
sys.path.insert(0, '/content')

# Set environment variables for the backend
import os
os.environ['DB_PATH'] = DB_PATH
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
os.environ['FEEDBACK_CSV'] = FEEDBACK_CSV
os.environ['STATS_JSON'] = STATS_JSON
os.environ['CORRECTION_DICT_PATH'] = CORRECTION_DICT_PATH
os.environ['CHECKPOINT_FILE'] = CHECKPOINT_FILE
os.environ['MODEL_CACHE'] = MODEL_CACHE
os.environ['PDF_PATH'] = PDF_PATH

# Start the API server with ngrok
from pyngrok import ngrok
import threading
import uvicorn

# Kill any existing tunnels
ngrok.kill()

# Create tunnel
public_url = ngrok.connect(8000, bind_tls=True)
print(f"API URL: {public_url}")
print(f"API Docs: {public_url}/docs")

# Start server in background
def run_server():
    # We'll use the backend app
    from backend.app import app
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("\n" + "="*60)
print("الخادم يعمل! انسخ الرابط التالي وألصقه في واجهة الويب:")
print(f"  {public_url}")
print("="*60)
print("\nافتح واجهة الويب في المتصفح وأدخل الرابط في الإعدادات.")
